In [11]:
#Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import joblib
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

MODEL_READY_PATH = "../logs_model_ready.parquet"
df_model = pd.read_parquet(MODEL_READY_PATH)

print("Modeling dataset shape:", df_model.shape)
display(df_model.head())

Modeling dataset shape: (13352, 17)


,game_session_id,move_index,played_by_encoded,ai_difficulty,board_X_count,board_O_count,board_empty_count,board_0,board_1,board_2,board_3,board_4,board_5,board_6,board_7,board_8,win_label_pov
0,0043a101-b212-405a-88d1-f7cc9e8be0b8,1,0,EASY,0,0,9,0,0,0,0,0,0,0,0,0,0.0
1,0043a101-b212-405a-88d1-f7cc9e8be0b8,2,1,HARD,1,0,8,1,0,0,0,0,0,0,0,0,1.0
2,0043a101-b212-405a-88d1-f7cc9e8be0b8,3,0,EASY,1,1,7,1,0,0,0,2,0,0,0,0,0.0
3,0043a101-b212-405a-88d1-f7cc9e8be0b8,4,1,HARD,2,1,6,1,1,0,0,2,0,0,0,0,1.0
4,0043a101-b212-405a-88d1-f7cc9e8be0b8,5,0,EASY,2,2,5,1,1,2,0,2,0,0,0,0,0.0


In [12]:
games = df_model["game_session_id"].unique()

train_games, test_games = train_test_split(
    games,
    test_size=0.30,
    random_state=42,
    shuffle=True,
)

train_mask = df_model["game_session_id"].isin(train_games)
test_mask = df_model["game_session_id"].isin(test_games)

train_df = df_model[train_mask].reset_index(drop=True)
test_df = df_model[test_mask].reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)


Train shape: (9398, 17)
Test shape:  (3954, 17)


## Define features and target

**Target**: `win_label_pov`
- 1.0 – player to move eventually wins,
- 0.0 – player to move eventually loses,
- 0.5 – draw.

This is a regression problem on [0, 1].
**Features** combine:
- board configuration (`board_0..board_8`),
- game phase (`move_index`),
- player identity (`played_by_encoded`),
- AI difficulty (`ai_difficulty`) – categorical.


In [13]:
target_col = "win_label_pov"

board_cols = [
    c for c in df_model.columns
    if c.startswith("board_") and c[len("board_"):].isdigit()
]
numeric_features = [
    "move_index",
    "played_by_encoded",
    "board_X_count",
    "board_O_count",
    "board_empty_count",
] + board_cols

categorical_features = ["ai_difficulty"]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train = train_df[numeric_features + categorical_features]
y_train = train_df[target_col]

X_test = test_df[numeric_features + categorical_features]
y_test = test_df[target_col]


Numeric features: ['move_index', 'played_by_encoded', 'board_X_count', 'board_O_count', 'board_empty_count', 'board_0', 'board_1', 'board_2', 'board_3', 'board_4', 'board_5', 'board_6', 'board_7', 'board_8']
Categorical features: ['ai_difficulty']


## Preprocessing pipeline
We use the same preprocessing for all models:

- numeric features -> `StandardScaler` (as in the lecture examples),
- categorical features (`ai_difficulty`) -> `OneHotEncoder`.

This `preprocessor` will be the first step in every `sklearn.Pipeline`.

In [14]:
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## Candidate models and hyperparameter grids

We consider three model families:

1. Ridge Regression – linear baseline, interpretable and stable.
2. RandomForestRegressor – non-linear model capturing interactions between cells on the board and move index.
3. XGBoost – gradient boosting model that builds trees sequentially, each correcting errors of previous ones, often achieving best performance.


Both are wrapped in a `Pipeline(preprocess -> model)` and tuned with `GridSearchCV`
using negative MSE as the scoring metric.

In [15]:
#1 Ridge regression pipeline + param grid

ridge_pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", Ridge())
    ]
)

ridge_param_grid = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0],
}

# 2 Random forest pipeline + param grid

rf_pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ))
    ]
)

rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, 8],
    "model__min_samples_leaf": [1, 3],
}
# 3 XGBoost pipeline + param grid

xgb_pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", XGBRegressor(
            random_state=42,
            n_jobs=-1
        ))
    ]
)

xgb_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
}


## GridSearchCV

We use cross-validation on the training set.
Scoring: `neg_mean_squared_error` (lower MSE = better), which is standard for regression and aligns with “how far predicted probability is from true win_label_pov”.

In [16]:
def run_grid_search(pipe, param_grid, X, y, model_name):
    print(f"\n=== Grid search for {model_name} ===")
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="neg_mean_squared_error",
        cv=5,
        n_jobs=-1,
        verbose=1,
        return_train_score=True,
    )
    grid.fit(X, y)
    print(f"Best params for {model_name}: {grid.best_params_}")
    print(f"Best CV MSE for {model_name}: {-grid.best_score_:.4f}")
    return grid


ridge_grid = run_grid_search(ridge_pipe, ridge_param_grid, X_train, y_train, "Ridge")
rf_grid = run_grid_search(rf_pipe, rf_param_grid, X_train, y_train, "RandomForest")
xgb_grid = run_grid_search(xgb_pipe, xgb_param_grid, X_train, y_train, "XGBoost")



=== Grid search for Ridge ===
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best params for Ridge: {'model__alpha': 10.0}
Best CV MSE for Ridge: 0.1026

=== Grid search for RandomForest ===
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best params for RandomForest: {'model__max_depth': 8, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
Best CV MSE for RandomForest: 0.0579

=== Grid search for XGBoost ===
Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best params for XGBoost: {'model__learning_rate': 0.05, 'model__max_depth': 7, 'model__n_estimators': 300, 'model__subsample': 0.8}
Best CV MSE for XGBoost: 0.0540


## Models comparison based on CV

We summarise the best cross-validated MSE for each model.
The one with lower MSE will be selected as the final model for deployment.

In [17]:
# results_cv = pd.DataFrame([
#     {
#         "model": "Ridge",
#         "best_params": ridge_grid.best_params_,
#         "cv_mse": -ridge_grid.best_score_,
#     },
#     {
#         "model": "RandomForest",
#         "best_params": rf_grid.best_params_,
#         "cv_mse": -rf_grid.best_score_,
#     },
# ])
#
# display(results_cv)

In [18]:
def summarize_grid_results(grid, model_name):
    """Create a tidy DataFrame with CV results for a given GridSearchCV."""
    res = pd.DataFrame(grid.cv_results_)
    # Keep only useful columns
    cols = [c for c in res.columns if c.startswith("param_")] + [
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
    ]
    res = res[cols].copy()
    # Convert neg MSE to positive MSE
    res["mean_test_MSE"] = -res["mean_test_score"]
    res["mean_train_MSE"] = -res["mean_train_score"]
    res["model"] = model_name
    return res


ridge_res = summarize_grid_results(ridge_grid, "Ridge")
rf_res = summarize_grid_results(rf_grid, "RandomForest")
xgb_res = summarize_grid_results(xgb_grid, "XGBoost")


all_cv_results = pd.concat([ridge_res, rf_res, xgb_res], ignore_index=True)

In [19]:
#  cd C:\Users\A\PycharmProjects\banditgames-ai\ML_winning_probability
# mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000

mlflow.set_experiment("win_probability")
print("Tracking URI:", mlflow.get_tracking_uri())
exp = mlflow.get_experiment_by_name("win_probability")
print("Experiment object:", exp)


Tracking URI: sqlite:///mlflow.db
Experiment object: <Experiment: artifact_location='file:///C:/Users/A/PycharmProjects/banditgames-ai/ML_winning_probability/mlruns/1', creation_time=1765204016698, experiment_id='1', last_update_time=1765204016698, lifecycle_stage='active', name='win_probability', tags={'mlflow.experimentKind': 'custom_model_development'}>


In [20]:
with mlflow.start_run(run_name="winprob_ridge_vs_rf_vs_xgb"):
    results_cv = pd.DataFrame([
        {
            "model": "Ridge",
            "best_params": ridge_grid.best_params_,
            "cv_mse": -ridge_grid.best_score_,
        },
        {
            "model": "RandomForest",
            "best_params": rf_grid.best_params_,
            "cv_mse": -rf_grid.best_score_,
        },
        {
            "model": "XGBoost",
            "best_params": xgb_grid.best_params_,
            "cv_mse": -xgb_grid.best_score_,
        },
    ])

    display(results_cv)

    ridge_res = summarize_grid_results(ridge_grid, "Ridge")
    rf_res    = summarize_grid_results(rf_grid, "RandomForest")
    xgb_res   = summarize_grid_results(xgb_grid, "XGBoost")
    all_cv_results = pd.concat([ridge_res, rf_res, xgb_res], ignore_index=True)
    display(all_cv_results)

    # log best CV MSE for all 3 models
    mlflow.log_metric("ridge_best_cv_mse", -ridge_grid.best_score_)
    mlflow.log_metric("rf_best_cv_mse",    -rf_grid.best_score_)
    mlflow.log_metric("xgb_best_cv_mse",   -xgb_grid.best_score_)

    all_grids = {
        "Ridge": ridge_grid,
        "RandomForest": rf_grid,
        "XGBoost": xgb_grid,
    }
    best_name = min(all_grids.keys(), key=lambda name: -all_grids[name].best_score_)
    best_grid = all_grids[best_name]

    print(f"Selected best model based on CV: {best_name}")
    print("Best params:", best_grid.best_params_)

    # best model + parameters
    mlflow.log_param("selected_model", best_name)
    for k, v in best_grid.best_params_.items():
        mlflow.log_param(f"{best_name}_{k}", v)

    y_pred_test = best_grid.predict(X_test)

    mse_test = mean_squared_error(y_test, y_pred_test)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    r2_test  = r2_score(y_test, y_pred_test)

    print("\nTest set performance of selected model:")
    print(f"  MSE: {mse_test:.4f}")
    print(f"  MAE: {mae_test:.4f}")
    print(f"  R² : {r2_test:.4f}")

    # log test metrics
    mlflow.log_metric("test_mse", mse_test)
    mlflow.log_metric("test_mae", mae_test)
    mlflow.log_metric("test_r2",  r2_test)

    # cv analyse for random forest
    rf_plot = rf_res.copy()
    rf_plot["max_depth"]    = rf_plot["param_model__max_depth"].astype(int)
    rf_plot["n_estimators"] = rf_plot["param_model__n_estimators"].astype(int)

    pivot = rf_plot.pivot_table(
        index="max_depth",
        columns="n_estimators",
        values="mean_test_MSE"
    )

    print("CV MSE for RandomForest (rows=max_depth, cols=n_estimators):")
    display(pivot)

    # cv results saved as artifacts for reports
    results_cv.to_csv("winprob_cv_summary.csv", index=False)
    all_cv_results.to_csv("winprob_all_cv_results.csv", index=False)
    pivot.to_csv("winprob_rf_pivot.csv")

    mlflow.log_artifact("winprob_cv_summary.csv")
    mlflow.log_artifact("winprob_all_cv_results.csv")
    mlflow.log_artifact("winprob_rf_pivot.csv")

    # sample predictions
    sample_idx      = X_test.sample(10, random_state=42).index
    sample_features = X_test.loc[sample_idx]
    sample_true     = y_test.loc[sample_idx]
    sample_pred     = best_grid.predict(sample_features)

    preview = sample_features.copy()
    preview["true_win_label_pov"] = sample_true.values
    preview["pred_win_prob"]      = sample_pred

    display(preview)

    preview.to_csv("winprob_sample_predictions.csv", index=False)
    mlflow.log_artifact("winprob_sample_predictions.csv")

    joblib.dump(best_grid, "winprob_best_model.joblib")
    mlflow.sklearn.log_model(best_grid, artifact_path="winprob_model")

,model,best_params,cv_mse
0,Ridge,{'model__alpha': 10.0},0.102574
1,RandomForest,"{'model__max_depth': 8, 'model__min_samples_le...",0.057854
2,XGBoost,"{'model__learning_rate': 0.05, 'model__max_dep...",0.053977


,param_model__alpha,mean_test_score,std_test_score,mean_train_score,mean_test_MSE,mean_train_MSE,model,param_model__max_depth,param_model__min_samples_leaf,param_model__n_estimators,param_model__learning_rate,param_model__subsample
0,0.01,-0.102575,0.007064,-0.101876,0.102575,0.101876,Ridge,NaN,NaN,NaN,NaN,NaN
1,0.10,-0.102575,0.007064,-0.101876,0.102575,0.101876,Ridge,NaN,NaN,NaN,NaN,NaN
2,1.00,-0.102575,0.007062,-0.101876,0.102575,0.101876,Ridge,NaN,NaN,NaN,NaN,NaN
3,10.00,-0.102574,0.007045,-0.101878,0.102574,0.101878,Ridge,NaN,NaN,NaN,NaN,NaN
4,NaN,-0.095712,0.007720,-0.093897,0.095712,0.093897,RandomForest,3.0,1.0,100.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
65,NaN,-0.054167,0.005495,-0.050465,0.054167,0.050465,XGBoost,7.0,NaN,100.0,0.1,1.0
66,NaN,-0.053990,0.005466,-0.050234,0.053990,0.050234,XGBoost,7.0,NaN,200.0,0.1,0.8
67,NaN,-0.054119,0.005569,-0.050165,0.054119,0.050165,XGBoost,7.0,NaN,200.0,0.1,1.0
68,NaN,-0.054037,0.005589,-0.050173,0.054037,0.050173,XGBoost,7.0,NaN,300.0,0.1,0.8


Selected best model based on CV: XGBoost
Best params: {'model__learning_rate': 0.05, 'model__max_depth': 7, 'model__n_estimators': 300, 'model__subsample': 0.8}

Test set performance of selected model:
  MSE: 0.0514
  MAE: 0.1392
  R² : 0.7189
CV MSE for RandomForest (rows=max_depth, cols=n_estimators):


n_estimators,100,200
max_depth,,
3,0.095712,0.095653
5,0.073882,0.073650
8,0.058156,0.058046


,move_index,played_by_encoded,board_X_count,board_O_count,board_empty_count,board_0,board_1,board_2,board_3,board_4,board_5,board_6,board_7,board_8,ai_difficulty,true_win_label_pov,pred_win_prob
3390,3,0,1,1,7,1,2,0,0,0,0,0,0,0,EASY,0.5,0.484578
2119,4,1,2,1,6,1,2,1,0,0,0,0,0,0,MEDIUM,0.0,0.539328
2507,8,1,4,3,2,2,1,0,1,2,0,1,2,1,HARD,0.5,0.500166
70,7,0,3,3,3,1,2,1,2,1,2,0,0,0,MEDIUM,1.0,1.000743
2610,4,1,2,1,6,1,1,2,0,0,0,0,0,0,HARD,1.0,0.901406
2547,4,1,2,1,6,1,2,1,0,0,0,0,0,0,MEDIUM,1.0,0.539328
2117,2,1,1,0,8,1,0,0,0,0,0,0,0,0,MEDIUM,0.0,0.400473
2694,3,0,1,1,7,1,2,0,0,0,0,0,0,0,EASY,0.5,0.484578
270,5,0,2,2,5,1,1,2,0,0,0,0,0,2,EASY,0.0,-0.000766
1270,2,1,1,0,8,1,0,0,0,0,0,0,0,0,MEDIUM,0.0,0.400473


2025/12/30 16:14:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
